# "Machine Learning \& Deep Learning Applications in Modern Power Systems"
## Department of Electrical Engineering - PMEC, Berhampur
### Instructor: Dr. Suryalok Dash 

# Day 2: Exercise Notebook
**Task:** Complete the missing code sections marked with `### YOUR CODE HERE ###`.

In [1]:
import torch
import torch.nn as nn
import numpy as np

# We provide pre-generated synthetic 2D data simulating solar panel scans. 
# Image size is 20x20.

num_samples = 800
img_size = 20
X_raw = np.random.rand(num_samples, img_size, img_size)
y_raw = np.random.randint(0, 2, num_samples) # 0 = Normal, 1 = Fault

# Task 1: A PyTorch CNN requires the shape (Batch, Channels, Height, Width).
# Reshape X_raw from (800, 20, 20) to (800, 1, 20, 20). Save it as X_reshaped.
X_reshaped = X_raw.reshape(num_samples, 1, img_size, img_size)

# Task 2: Convert X_reshaped into a float32 PyTorch tensor named 'X_tensor'
X_tensor = torch.tensor(X_reshaped, dtype=torch.float32)

# Task 3: Convert y_raw into a torch.long (integer) tensor named 'y_tensor'
y_tensor = torch.tensor(y_raw, dtype=torch.long)

In [2]:
class SolarCNN(nn.Module):
    def __init__(self):
        super(SolarCNN, self).__init__()
        
        # Task 4: Define a Conv2d layer named 'self.conv1'. 
        # Set in_channels=1, out_channels=16, and kernel_size=3. Add padding=1.
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        
        # Task 5: Define a ReLU activation function named 'self.relu'
        self.relu = nn.ReLU()
        
        # Task 6: Define a MaxPool2d layer named 'self.pool'. 
        # Set kernel_size=2 and stride=2.
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)
        
        # 20x20 image -> pooled by 2 -> becomes 10x10.
        # Flattened size: 16 channels * 10 height * 10 width = 1600
        self.fc1 = nn.Linear(1600, 2) 

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu(x)
        x = self.pool(x)
        # We flatten the tensor for the dense layer here
        x = x.view(x.size(0), -1) 
        x = self.fc1(x)
        return x

solar_model = SolarCNN()

In [3]:
# Task 7: Define the loss function as CrossEntropyLoss. Name it 'criterion'.
criterion = nn.CrossEntropyLoss()

# Task 8: Initialize an Adam optimizer. Pass it 'solar_model.parameters()'. Name it 'optimizer'.
# Set the learning rate (lr) to 0.005.
optimizer = torch.optim.Adam(solar_model.parameters(), lr=0.005)

# Task 9: Split the tensors into train and test manually.
# Take the first 600 samples of X_tensor for 'X_train' and the remaining 200 for 'X_test'
X_train = X_tensor[:600]
X_test  = X_tensor[600:]
y_train = y_tensor[:600]
y_test  = y_tensor[600:]

In [4]:
epochs = 15

for epoch in range(epochs):
    # Task 10: Set the model to training mode
    solar_model.train()
    
    # Task 11: Zero the gradients of the optimizer
    optimizer.zero_grad()
    
    outputs = solar_model(X_train)
    loss = criterion(outputs, y_train)
    
    # Task 12: Perform the backward pass AND step the optimizer to update weights
    loss.backward()
    optimizer.step()
    
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

Epoch 5, Loss: 1.6281
Epoch 10, Loss: 1.0688
Epoch 15, Loss: 0.7504


In [5]:
# Task 13: Set the model to evaluation mode
solar_model.eval()

with torch.no_grad():
    # Task 14: Pass X_test through the model to get 'test_outputs'
    test_outputs = solar_model(X_test)
    
    # This line extracts the predicted class (0 or 1) from the raw outputs
    _, predicted_classes = torch.max(test_outputs, 1)
    
    # Task 15: Calculate the accuracy. Compare 'predicted_classes' to 'y_test'.
    # Convert to float and take the mean. Multiply by 100 to get a percentage.
    accuracy = (predicted_classes == y_test).float().mean() * 100
    
    print(f"Test Accuracy: {accuracy.item():.2f}%")

Test Accuracy: 43.50%
